# 🌊 Ocean Dynamics: Environmentally Driven Biomass Simulation

### Module 3 of the SquidStock Analytics Series

## 🧭 Problem Framing and Decision Context

Standardized catch per unit effort (CPUE) provides a useful index of relative fishing success, but it does not directly measure total stock biomass. This distinction is especially important for short-lived, highly mobile species such as *Illex argentinus*, where catch rates may also be influenced by aggregation, migration, fleet targeting, environmental conditions, and changing catchability.

This notebook extends earlier CPUE analyses by asking:

* How might modelled squid biomass respond to environmental variability?
* How does a gradual warming scenario alter the simulated biomass trajectory?
* How closely does observed CPUE track biomass generated by the model?
* How sensitive are interpretations to assumptions about growth, catchability, fishing effort, and thermal conditions?

Using an Environmentally Dependent Surplus Production Model (EDSPM), the notebook simulates biomass under baseline and warming assumptions. The model is intended for transparent scenario exploration, monitoring discussions, and risk-aware decision support. It is not a formal stock assessment or regulatory biomass forecast.

---

This notebook reproduces the main analytical logic of the interactive Streamlit simulator using notebook-based Plotly visualizations and the same default model settings.

## ✅ Executive Summary

* **Baseline scenario:** Simulates biomass using temperature-dependent growth, a chlorophyll-a productivity modifier, carrying-capacity constraints, fishing effort, catchability, and modest stochastic variation in growth.
* **Warming scenario:** Applies a gradual temperature increase and compares the resulting biomass trajectory with the equivalent baseline period.
* **CPUE comparison:** Evaluates CPUE against both baseline and warming biomass outputs. Any weak or negative relationship should be interpreted as divergence from modelled biomass, not proof that CPUE is unrelated to true stock abundance.
* **Seasonal-data limitation:** The analysis uses January–June observations because those months were consistently available across the study period. The resulting sequence does not represent complete year-round population dynamics.
* **Decision-support value:** The notebook demonstrates how CPUE, environmental conditions, fishing effort, and modelled biomass can provide different signals. These outputs can support scenario comparison and monitoring design, but not formal quota setting or regulatory stock determination.

---

## 🧭 Overview

The EDSPM links environmental conditions, density dependence, and fishing removals through the following components:

* **Biomass (Nₜ):** simulated squid biomass at time *t*
* **Temperature-dependent growth rate (rₜ):** changes nonlinearly with sea surface temperature
* **Chlorophyll-a productivity modifier (Eₚᵣₒd,ₜ):** a simplified indicator of relative food-web productivity used in the baseline simulation
* **Thermal suitability:** a diagnostic indicator showing how close temperature is to the assumed thermal optimum
* **Carrying capacity (K):** the assumed upper ecological limit on biomass
* **Fishing removals:** represented using catchability, effort, and available biomass

Temperature and chlorophyll-a are treated as separate pathways in the baseline simulation. Sea surface temperature controls the nonlinear growth-rate response, while normalized chlorophyll-a modifies potential biomass production. This avoids placing temperature inside two separate growth multipliers.

The warming simulation retains the same nonlinear temperature-growth relationship and fishing-removal equation. In the current app implementation, thermal suitability is used as a diagnostic indicator in the warming panel and is not multiplied into growth a second time.

This module moves beyond standardized abundance indices by providing a transparent population-level scenario model for comparing baseline and warming assumptions.

---

## ⚙️ About the EDSPM

For the baseline simulation, modelled production at time *t* is represented as:

*Pₜ = rₜ × Eₚᵣₒd,ₜ × Nₜ × (1 − Nₜ / K)*

Where:

* **Pₜ:** modelled biomass production at time *t*
* **Nₜ:** modelled biomass at time *t*
* **K:** assumed carrying capacity
* **rₜ:** temperature-dependent growth rate
* **Eₚᵣₒd,ₜ:** chlorophyll-a-based productivity modifier

Fishing removals are represented separately:

*Hₜ = q × Effortₜ × Nₜ*

The biomass update is:

*Nₜ₊₁ = max(Nₜ + Pₜ − Hₜ, 0)*

Where:

* **Hₜ:** modelled fishing removal at time *t*
* **q:** catchability coefficient
* **Effortₜ:** fishing effort at time *t*

The chlorophyll-a productivity modifier is represented as:

*Eₚᵣₒd,ₜ = 0.7 + 0.3 × normalized Chl-aₜ*

This constrains the productivity modifier to approximately 0.7–1.0, allowing chlorophyll-a to influence production without forcing growth to zero during low-productivity observations.

### 🌡️ Nonlinear Temperature-Dependent Growth

Growth varies with sea surface temperature according to a Gaussian thermal-response curve:

*rₜ = r₀ × exp[−(SSTₜ − Tₒₚₜ)² / (2σₜ²)]*

Interpretation:

* Growth is highest when SST is close to the assumed thermal optimum.
* Growth decreases when temperature moves either below or above the optimum.
* The direction of a warming response depends on whether warming moves temperatures toward or away from the optimum.
* This thermal-response function is nonlinear even though a separate linear association line is later used to summarize the CPUE–biomass scatter plot.

---

## 📊 Default Scenario Settings

| Parameter                        |       Default value | Interpretation                                        |
| -------------------------------- | ------------------: | ----------------------------------------------------- |
| Carrying capacity, **K**         |      5,000,000 tons | Assumed upper biomass limit                           |
| Initial biomass, **N₀**          |      3,000,000 tons | Starting biomass for the simulation                   |
| Maximum growth parameter, **r₀** | 0.03 per model step | Upper growth-rate parameter before thermal adjustment |
| Thermal optimum, **Tₒₚₜ**        |               12 °C | Temperature associated with maximum modelled growth   |
| Thermal tolerance, **σₜ**        |                3 °C | Width of the thermal-response curve                   |
| Catchability, **q**              |            2 × 10⁻⁴ | Fishing-removal efficiency per unit effort            |
| Baseline simulations             |                 500 | Number of stochastic baseline trajectories            |
| Warming simulations              |               1,000 | Number of stochastic warming trajectories             |

These values are exploratory scenario defaults. They provide a stable reference configuration but should not be interpreted as formally estimated stock parameters.

---

## 📐 Scenario Plausibility and Parameter Checks

The model has not been formally calibrated against an independently observed biomass time series. Its parameters are treated as adjustable scenario assumptions.

A simple starting-biomass plausibility check is:

*N₀ ≥ maximum observed catch / illustrative target exploitation fraction*

This check asks whether the assumed starting biomass is large enough to support the largest observed catch without implying an implausibly high removal fraction.

The calculation is a screening rule only. It is not a formal stock-assessment calibration, biological reference point, or estimate of sustainable exploitation.

The notebook calculates the maximum observed catch directly from the loaded data rather than relying on a hard-coded historical value.

---

## 🎲 Handling Uncertainty with Monte Carlo Simulation

The notebook follows the current app’s Monte Carlo structure:

1. The baseline simulation runs 500 biomass trajectories.
2. The warming simulation runs 1,000 biomass trajectories.
3. Each trajectory receives a small multiplicative growth perturbation drawn around 1.0 with a standard deviation of 0.05.
4. Model parameters remain fixed at the selected scenario values.
5. Mean biomass and central 95% simulation intervals are calculated across trajectories.
6. CPUE receives a separate illustrative 10% observation-noise assumption for the sensitivity comparison.

The shaded bands are **Monte Carlo simulation intervals**, not formal statistical confidence intervals. They show the spread generated by the selected stochastic-growth assumptions.

Monte Carlo simulation helps evaluate how model outputs vary under those assumptions. It does not distinguish true ecological signals from noise unless the uncertainty structure itself has been empirically validated.

---

## 📉 Scenario-Interpretation Threshold

For communication purposes, a ±5% difference from the baseline is used as an exploratory interpretation band:

* Between −5% and +5%: broadly stable
* Above +5%: meaningfully higher under the selected scenario
* Below −5%: meaningfully lower under the selected scenario

This is a practical scenario-screening heuristic. It is not a validated biological threshold, management reference point, or regulatory decision rule.

---

## 🌊 Environmental Data Notes

* SST is calculated from vessel-associated temperature observations.
* Chlorophyll-a is represented using monthly environmental data.
* Only January–June observations are included consistently.
* SST influences growth through the nonlinear thermal-response function.
* Chlorophyll-a is normalized and converted into a 0.7–1.0 productivity modifier for the baseline simulation.
* Thermal suitability is retained as a diagnostic indicator in the warming scenario.
* SST and chlorophyll-a originate from different spatial and temporal resolutions and should be interpreted cautiously.

---

## ⚠️ Model Limitations and Interpretation Boundaries

* The EDSPM is an exploratory scenario model, not a formally fitted stock-assessment model.
* Biomass values are simulated and should not be treated as observed or regulatory biomass estimates.
* Only January–June observations are represented.
* Transitions between June and the following January do not represent complete continuous population dynamics.
* The model does not explicitly represent recruitment, spawning, natural mortality, migration, life stage, age structure, ocean currents, prey dynamics, fleet redistribution, or spatial stock structure.
* A single thermal optimum is applied across all observations.
* Catchability is represented as a constant coefficient.
* Baseline and warming simulations currently differ in how the chlorophyll-a productivity modifier enters the production calculation.
* Monte Carlo intervals reflect selected stochastic assumptions and are not formal confidence intervals.
* CPUE is compared with modelled biomass, not independently measured true biomass.
* Warming responses depend on the starting temperatures, thermal optimum, duration, catchability, and other model assumptions.

---

## 🧩 Notebook Purpose

This notebook converts the Streamlit simulator into a transparent and reproducible analytical document. It allows users to:

* inspect the model equations and assumptions;
* reproduce the default baseline and warming scenarios;
* explore nonlinear thermal-growth responses;
* compare CPUE with both baseline and warming biomass;
* examine simulation uncertainty;
* and identify limitations relevant to fisheries decision support.

The following cells load the data, preprocess environmental variables, simulate biomass, visualize baseline and warming scenarios, and compare CPUE with modelled biomass.

In [10]:
# Cell 2: imports and helpers
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import kaleido
import matplotlib.pyplot as plt
import math
import os
from IPython.display import display, Markdown

# Helper: normalize series to 0-1 (avoid divide-by-zero)
def normalize(series):
    mn, mx = series.min(), series.max()
    if mx - mn == 0:
        return series * 0.0
    return (series - mn) / (mx - mn)


In [11]:
# Cell 3: load data (adjust path to your environment)
# This mirrors the Streamlit load_data() function.
data_path = "../data/Final_dataset_imputed.csv"  # adjust if needed

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Data file not found at {data_path} — update `data_path` accordingly.")

df_raw = pd.read_csv(data_path)
# restrict to Jan-Jun (1..6)
df_raw = df_raw[df_raw["Month"].between(1, 6)].copy()

# convert catch to tons
df_raw["SqCatch_tons"] = df_raw["SqCatch_Kg"] / 1000.0

# create a trip id similar to VesselDay
df_raw["VesselDay"] = df_raw["CTNO"].astype(str) + "_" + df_raw["Year"].astype(str) + "_" + df_raw["Month"].astype(str) + "_" + df_raw["Day"].astype(str)

# aggregate by trip then month (mirrors original)
trip_cpue = (
    df_raw.groupby(["Year", "Month", "CTNO", "VesselDay"], as_index=False)
    .agg(DayCatch_tons=("SqCatch_tons", "sum"))
)

monthly_summary = (
    trip_cpue.groupby(["Year", "Month"], as_index=False)
    .agg(
        TotalCatch_tons=("DayCatch_tons", "sum"),
        VesselDays=("VesselDay", "count")
    )
)

monthly_summary["CPUE_tons"] = monthly_summary["TotalCatch_tons"] / monthly_summary["VesselDays"]

env_features = (
    df_raw.groupby(["Year", "Month"], as_index=False)
    .agg(SST=("WaterTemp", "mean"), ChlA=("Chlor_a_mg_m3", "mean"))
)

df_monthly = monthly_summary.merge(env_features, on=["Year", "Month"]).sort_values(["Year", "Month"]).reset_index(drop=True)

# CPUE index within-year (as used originally)
df_monthly["CPUE_index"] = df_monthly.groupby("Year")["CPUE_tons"].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if (x.max() - x.min())!=0 else 0)

# effort weight (as in your script)
df_monthly["Effort_weight"] = df_monthly["VesselDays"] / df_monthly["VesselDays"].max()

print("Loaded monthly data — rows:", len(df_monthly))
df_monthly.head()


# ============================================================
# Model parameter defaults
# These values are exploratory scenario assumptions.
# ============================================================

K = 5_000_000
N0 = 3_000_000
r0 = 0.03
T_opt = 12.0
sigma_T = 3.0
q = 2e-4

num_sim = 500

print("Model parameters loaded:")
print(f"K = {K:,.0f} tons")
print(f"N0 = {N0:,.0f} tons")
print(f"r0 = {r0:.3f} per time step")
print(f"T_opt = {T_opt:.1f} °C")
print(f"sigma_T = {sigma_T:.1f} °C")
print(f"q = {q:.2e}")

Loaded monthly data — rows: 114
Model parameters loaded:
K = 5,000,000 tons
N0 = 3,000,000 tons
r0 = 0.030 per time step
T_opt = 12.0 °C
sigma_T = 3.0 °C
q = 2.00e-04


### Chlorophyll-a Productivity Modifier (`E_env`)

The baseline simulation converts chlorophyll-a into a normalized productivity modifier:

- Chlorophyll-a is normalized to a 0–1 range.
- The normalized value is transformed to a 0.7–1.0 multiplier.
- SST is not included in this modifier because temperature already affects growth through the nonlinear `r_t` function.

The implemented modifier is:

$E_{env,t} = 0.7 + 0.3 \times \text{normalized Chl-a}_t$

This prevents low chlorophyll-a observations from reducing the productivity multiplier to zero.

In [ ]:

# ============================================================
# ENVIRONMENTAL PREPROCESSING
# Matches preprocess_env() in the Streamlit app
# ============================================================

df = df_monthly.copy().reset_index(drop=True)

chla_min = df["ChlA"].min()
chla_max = df["ChlA"].max()

if chla_max == chla_min:
    df["E_env"] = 1.0
else:
    # Chlorophyll-a-only productivity modifier.
    # Temperature is handled separately through r_t.
    df["E_env"] = (
        (df["ChlA"] - chla_min)
        / (chla_max - chla_min)
    )

# Prevent normalized productivity from reaching zero
df["E_env"] = 0.7 + 0.3 * df["E_env"]

df[
    [
        "Year",
        "Month",
        "SST",
        "ChlA",
        "E_env"
    ]
].head()

,Year,Month,SST,ChlA,E_env
0,2000,1,13.668258,0.903811,0.752896
1,2000,2,13.237296,1.546665,0.799329
2,2000,3,12.107456,0.690164,0.737464
3,2000,4,10.106503,0.591688,0.730351
4,2000,5,8.798140,0.442545,0.719578


### Monte Carlo Baseline EDSPM

The baseline model runs 500 stochastic biomass trajectories using the same default settings as the Streamlit app.

For each model step:

- temperature determines the nonlinear growth rate;
- chlorophyll-a modifies potential production;
- carrying capacity constrains density-dependent growth;
- catchability and vessel-days determine fishing removals;
- and a small 5% multiplicative perturbation introduces variation among simulations.

All simulations begin from the same initial biomass. The model parameters themselves are not randomly perturbed in the current implementation.

The output includes:

- mean modelled biomass;
- lower and upper central 95% simulation bounds;
- and the temperature-dependent growth rate.

In [ ]:
# ============================================================
# MONTE CARLO BASELINE
# Matches run_baseline_simulation() in the Streamlit app
# ============================================================

num_sim = 500
T = len(df)

r_t = r0 * np.exp(
    -((df["SST"] - T_opt) ** 2)
    / (2 * sigma_T**2)
)

E_env = df["E_env"].to_numpy()
E_eff = df["VesselDays"].to_numpy()

biomass = np.zeros((num_sim, T))
biomass[:, 0] = N0

# Fixed seed makes notebook output reproducible.
# The Streamlit app currently does not use a fixed seed.
np.random.seed(42)

for t in range(1, T):
    N_prev = biomass[:, t - 1]

    noise = np.random.normal(
        1.0,
        0.05,
        size=num_sim
    )

    growth = (
        r_t.iloc[t]
        * E_env[t]
        * N_prev
        * (1 - N_prev / K)
        * noise
    )

    catch_loss = (
        q
        * E_eff[t]
        * N_prev
    )

    biomass[:, t] = np.maximum(
        N_prev + growth - catch_loss,
        0
    )

df_mc = df.copy().reset_index(drop=True)

df_mc["Biomass_mean"] = biomass.mean(axis=0)

df_mc["Biomass_CI_lower"] = np.percentile(
    biomass,
    2.5,
    axis=0
)

df_mc["Biomass_CI_upper"] = np.percentile(
    biomass,
    97.5,
    axis=0
)

df_mc["r_t"] = r_t

# Same caps used by the app
df_mc["Biomass_mean"] = np.minimum(
    df_mc["Biomass_mean"],
    1.2 * K
)

df_mc["Biomass_CI_upper"] = np.minimum(
    df_mc["Biomass_CI_upper"],
    1.2 * K
)

df_mc[
    [
        "Year",
        "Month",
        "Biomass_mean",
        "Biomass_CI_lower",
        "Biomass_CI_upper",
        "r_t"
    ]
].head()

,Year,Month,Biomass_mean,Biomass_CI_lower,Biomass_CI_upper,r_t
0,2000,1,3.000000e+06,3.000000e+06,3.000000e+06,0.025702
1,2000,2,3.009039e+06,3.006628e+06,3.011693e+06,0.027554
2,2000,3,3.016916e+06,3.013477e+06,3.020438e+06,0.029981
3,2000,4,3.020414e+06,3.016136e+06,3.024503e+06,0.024582
4,2000,5,3.016317e+06,3.012126e+06,3.020610e+06,0.016973


### Baseline Biomass, Simulation Interval, Growth Rate, and Fishing Pressure

This figure summarizes the baseline EDSPM under the default assumptions.

1. **Mean modelled biomass**  
   The teal line shows the average biomass across 500 Monte Carlo trajectories.

2. **Central 95% simulation interval**  
   The shaded band shows the central range of biomass outcomes produced by the 5% stochastic growth assumption. It is not a formal confidence interval.

3. **Temperature-dependent growth rate**  
   The orange line shows how SST affects growth relative to the assumed thermal optimum. Growth is strongest near the optimum and decreases on either side.

The cell also reports two fishing-pressure indicators:

- **Observed catch divided by simulated biomass:** a descriptive screening ratio.
- **Modelled exploitation rate:** the fishing-removal proportion generated by the model’s catchability and effort assumptions.

Neither quantity should be interpreted as a formally estimated stock exploitation rate.

In [ ]:

# ============================================================
# BASELINE PLOT AND EXPLOITATION INDICATORS
# ============================================================

mean_catch = df_mc["TotalCatch_tons"].mean()
mean_biomass = df_mc["Biomass_mean"].mean()

# Observed catch divided by simulated biomass
exploitation_rate = (
    mean_catch / mean_biomass
    if mean_biomass > 0
    else np.nan
)

# Modelled harvest applied by the simulation equation
df_mc["ModelHarvest_tons"] = (
    q
    * df_mc["VesselDays"]
    * df_mc["Biomass_mean"]
)

model_exploitation_rate = (
    df_mc["ModelHarvest_tons"].mean()
    / mean_biomass
    if mean_biomass > 0
    else np.nan
)

month_x = np.arange(1, len(df_mc) + 1)

fig = go.Figure()

# Mean biomass
fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_mc["Biomass_mean"],
        mode="lines+markers",
        name="Biomass (mean)",
        line=dict(color="teal"),
        customdata=np.column_stack(
            (
                df_mc["Biomass_mean"] / 1_000_000,
                df_mc["SST"],
                df_mc["ChlA"],
                df_mc["Year"],
                df_mc["Month"],
                df_mc["VesselDays"]
            )
        ),
        hovertemplate=(
            "Observation: %{x}<br>"
            "Biomass: %{customdata[0]:.2f} million tons<br>"
            "SST: %{customdata[1]:.2f}°C<br>"
            "ChlA: %{customdata[2]:.3f} mg/m³<br>"
            "Year: %{customdata[3]:.0f}<br>"
            "Month: %{customdata[4]:.0f}<br>"
            "Effort: %{customdata[5]:.0f} vessel-days"
            "<extra></extra>"
        )
    )
)

# Upper uncertainty boundary
fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_mc["Biomass_CI_upper"],
        mode="lines",
        line=dict(color="lightgray"),
        showlegend=False,
        hovertemplate=(
            "Observation: %{x}<br>"
            "Upper 95% simulation interval: %{y:,.0f} tons"
            "<extra></extra>"
        )
    )
)

# Lower uncertainty boundary and ribbon
fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_mc["Biomass_CI_lower"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(211,211,211,0.25)",
        line=dict(color="lightgray"),
        name="95% simulation interval",
        hovertemplate=(
            "Observation: %{x}<br>"
            "Lower 95% simulation interval: %{y:,.0f} tons"
            "<extra></extra>"
        )
    )
)

# Temperature-dependent growth rate
fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_mc["r_t"],
        mode="lines+markers",
        name="Growth rate r_t",
        line=dict(
            color="orange",
            dash="dot"
        ),
        yaxis="y2",
        customdata=np.column_stack(
            (
                df_mc["SST"],
                df_mc["Year"],
                df_mc["Month"],
                df_mc["E_env"]
            )
        ),
        hovertemplate=(
            "Observation: %{x}<br>"
            "Growth rate r_t: %{y:.4f}<br>"
            "SST: %{customdata[0]:.2f}°C<br>"
            "Year: %{customdata[1]:.0f}<br>"
            "Month: %{customdata[2]:.0f}<br>"
            "Productivity modifier: %{customdata[3]:.3f}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=(
        "Baseline Biomass Simulation + "
        "Temperature-dependent Growth Rate"
    ),
    xaxis_title="Time sequence (January–June observations)",
    yaxis=dict(
        title="Biomass (tons, Monte Carlo mean)",
        side="left"
    ),
    yaxis2=dict(
        title="Growth rate r_t",
        overlaying="y",
        side="right"
    ),
    height=520,
    template="plotly_dark",
    hovermode="x unified"
)

fig.show()

from pathlib import Path

output_dir = Path("../outputs/EDSPM")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir
    / "temperature_dependent_growth_rate.png"
)

try:
    fig.write_image(str(output_path))
    print(f"Saved plot to: {output_path}")
except Exception as error:
    print(f"Static image export skipped: {error}")

print(
    "Observed catch / simulated biomass: "
    f"{exploitation_rate:.2%}"
)

print(
    "Modelled exploitation rate: "
    f"{model_exploitation_rate:.2%}"
)

Saved plot to: ..\outputs\EDSPM\temperature_dependent_growth_rate.png
Observed catch / simulated biomass: 0.61%
Modelled exploitation rate: 0.54%


## 1️⃣ Baseline Simulation Insights

### Temperature-Dependent Growth and Biomass

Under the default configuration, the full baseline simulation shows a clear upward biomass trajectory, increasing from approximately **3.0 million tonnes to around 3.6 million tonnes** across the available January–June observation sequence.

This represents an increase of roughly **20%** over the complete simulated sequence. The result therefore indicates sustained positive net biomass growth under the default assumptions rather than a broadly stable stock.

The trajectory reflects the combined effects of:

* temperature-dependent growth;
* the chlorophyll-a productivity modifier;
* density dependence through carrying capacity;
* fishing effort;
* catchability;
* and modest stochastic variation in growth.

The orange growth-rate line changes through time according to how closely SST aligns with the assumed thermal optimum. Growth strengthens when SST approaches the optimum and weakens when temperature moves away from it.

Because the thermal response is nonlinear, the model does not assume that continuously warmer water always increases biomass. Growth would eventually decline if temperatures moved beyond the favourable thermal range.

### Fishing-pressure interpretation

Modelled fishing removals remain light under the default catchability setting. As a result, environmental growth and productivity generally exceed simulated harvest losses, allowing biomass to increase through the sequence.

### Decision-support interpretation

The baseline provides an increasing reference trajectory against which warming and CPUE signals can be compared.

However, the increase should not be interpreted as an observed stock recovery or formal biomass forecast. It is the output of the selected model assumptions, including the default initial biomass, carrying capacity, growth parameters, environmental modifiers, effort, and catchability.

Because only January–June observations are available, the trajectory represents a sequence of early-season observations across years rather than complete continuous year-round population dynamics.

## 2️⃣ Warming Scenario Insights

### Simulated Biomass Under Baseline and +2°C Warming

The warming analysis compares the default baseline with a gradual **+2°C scenario** over the selected 24-observation period.

Both trajectories begin near **3.0 million tonnes**. By the end of the comparison period:

* baseline biomass reaches approximately **3.07 million tonnes**;
* warming biomass reaches approximately **3.20 million tonnes**.

The warming trajectory therefore ends approximately **130,000 tonnes above the baseline**, equivalent to an endpoint difference of roughly **4%**.

The warmed trajectory itself rises by approximately **6–7%** from its starting biomass. However, not all of that increase should be attributed to warming because the baseline trajectory also increases. The warming effect is represented by the difference between the two trajectories, not by the total rise in the warming line alone.

### Panel 1 — Simulated Biomass Under Two Scenarios

The first panel shows a visible separation between the baseline and warming trajectories.

Although both scenarios increase, the warming trajectory rises more strongly and finishes clearly above the baseline. The difference is meaningful visually and in absolute biomass, but remains moderate relative to the approximately 3-million-tonne starting stock.

### Panel 2 — Percentage Change Due to Warming

The second panel isolates the warming effect by calculating the percentage difference between warming biomass and the corresponding baseline biomass.

Under the default settings, this difference develops progressively and reaches approximately **5% by the end of the scenario period**.

This is more informative than comparing the warming endpoint only with its own starting value because it removes the biomass increase that also occurs under baseline conditions.

### Panel 3 — Thermal Suitability and Effort

The third panel shows thermal suitability alongside vessel-day effort.

Under the default +2°C configuration, warming generally moves temperatures closer to the assumed thermal optimum during part of the selected period. This improves the modelled growth-rate response and contributes to the separation between the warming and baseline biomass trajectories.

Fishing effort remains represented as a separate removal pressure. Under the default catchability value, those removals are not strong enough to offset the additional modelled growth.

### Decision-support interpretation

The default +2°C scenario produces a **moderate positive biomass response relative to baseline**, rather than merely a negligible change.

However, this is a conditional scenario result—not evidence that climate warming is generally beneficial to *Illex argentinus*. The outcome depends on the starting SST values, the assumed 12°C thermal optimum, scenario duration, catchability, effort, carrying capacity, and simplified population structure.

The model is best interpreted as identifying a temporary **favourable warming window**: warming can increase growth while temperatures move toward the assumed optimum, but additional warming would eventually reduce growth once temperatures move beyond that optimum.

In [24]:
# ============================================================
# WARMING SCENARIO
# ============================================================

delta_T = 2.0
duration = min(24, len(df_mc))
show_baseline = True

df_warm = (
    df_mc.copy()
    .iloc[:duration]
    .reset_index(drop=True)
)

# Apply the same linear warming increment as the app
df_warm["SST"] = (
    df_warm["SST"]
    + np.linspace(
        0,
        delta_T,
        len(df_warm)
    )
)

# Warming temperature-dependent growth
df_warm["r_t"] = r0 * np.exp(
    -((df_warm["SST"] - T_opt) ** 2)
    / (2 * sigma_T**2)
)

# Baseline and warming thermal suitability
df_mc["ThermalSuitability"] = np.exp(
    -((df_mc["SST"] - T_opt) ** 2)
    / (2 * sigma_T**2)
)

df_warm["ThermalSuitability"] = np.exp(
    -((df_warm["SST"] - T_opt) ** 2)
    / (2 * sigma_T**2)
)

# Used only for plotting in the app
df_mc["EnvIndex"] = df_mc["ThermalSuitability"]
df_warm["EnvIndex"] = df_warm["ThermalSuitability"]

df_warm["Effort"] = df_warm["VesselDays"]
df_mc["Effort"] = df_mc["VesselDays"]

n_sim = 1000
biomass_sim = np.zeros(
    (n_sim, len(df_warm))
)

# Optional fixed seed for reproducible notebook output
np.random.seed(42)

for i in range(n_sim):

    # Same starting point as app
    N = float(
        df_mc["Biomass_mean"].iloc[0]
    )

    for t in range(len(df_warm)):

        r_t_current = float(
            df_warm["r_t"].iloc[t]
        )

        effort_current = float(
            df_warm["Effort"].iloc[t]
        )

        noise = np.random.normal(
            1.0,
            0.05
        )

        # IMPORTANT:
        # This mirrors the current app.
        # EnvIndex is diagnostic only and is not multiplied here.
        growth = (
            r_t_current
            * N
            * (1 - N / K)
            * noise
        )

        harvest = (
            q
            * effort_current
            * N
        )

        N = max(
            N + growth - harvest,
            0.0
        )

        biomass_sim[i, t] = N

df_warm["Biomass_mean"] = (
    biomass_sim.mean(axis=0)
)

df_warm["Biomass_CI_lower"] = np.percentile(
    biomass_sim,
    2.5,
    axis=0
)

df_warm["Biomass_CI_upper"] = np.percentile(
    biomass_sim,
    97.5,
    axis=0
)

baseline_slice = (
    df_mc["Biomass_mean"]
    .iloc[:duration]
    .to_numpy()
)

with np.errstate(
    divide="ignore",
    invalid="ignore"
):
    df_warm["Biomass_change_pct"] = (
        100
        * (
            df_warm["Biomass_mean"].to_numpy()
            - baseline_slice
        )
        / (baseline_slice + 1e-12)
    )


# Three-panel warming figure
month_x = np.arange(1, len(df_warm) + 1)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=(
        "Simulated Biomass",
        "% Change in Biomass",
        "Thermal Suitability Index and Effort"
    )
)

# ------------------------------------------------------------
# Panel 1: Baseline and warming biomass
# ------------------------------------------------------------

if show_baseline:
    baseline_biomass = (
        df_mc["Biomass_mean"]
        .iloc[:duration]
        .to_numpy()
    )

    fig.add_trace(
        go.Scatter(
            x=month_x,
            y=baseline_biomass,
            mode="lines",
            name="Baseline (mean)",
            line=dict(color="blue"),
            customdata=np.column_stack(
                (
                    baseline_biomass / 1_000_000,
                    df_mc["SST"].iloc[:duration],
                    df_mc["Year"].iloc[:duration],
                    df_mc["Month"].iloc[:duration]
                )
            ),
            hovertemplate=(
                "Observation: %{x}<br>"
                "Baseline biomass: %{customdata[0]:.2f} million tons<br>"
                "Baseline SST: %{customdata[1]:.2f}°C<br>"
                "Year: %{customdata[2]:.0f}<br>"
                "Month: %{customdata[3]:.0f}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1
    )

warming_biomass = df_warm["Biomass_mean"].to_numpy()

fig.add_trace(
    go.Scatter(
        x=month_x,
        y=warming_biomass,
        mode="lines+markers",
        name="Warming (mean)",
        line=dict(color="orange"),
        customdata=np.column_stack(
            (
                warming_biomass / 1_000_000,
                df_warm["SST"],
                df_mc["SST"].iloc[:duration],
                df_warm["SST"]
                - df_mc["SST"].iloc[:duration].to_numpy(),
                df_warm["Year"],
                df_warm["Month"]
            )
        ),
        hovertemplate=(
            "Observation: %{x}<br>"
            "Warming biomass: %{customdata[0]:.2f} million tons<br>"
            "Warmed SST: %{customdata[1]:.2f}°C<br>"
            "Baseline SST: %{customdata[2]:.2f}°C<br>"
            "Temperature increment: %{customdata[3]:.2f}°C<br>"
            "Year: %{customdata[4]:.0f}<br>"
            "Month: %{customdata[5]:.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_warm["Biomass_CI_upper"],
        mode="lines",
        line=dict(
            color="orange",
            dash="dot"
        ),
        showlegend=False,
        hovertemplate=(
            "Observation: %{x}<br>"
            "Upper 95% simulation interval: %{y:,.0f} tons"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_warm["Biomass_CI_lower"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(255,165,0,0.20)",
        line=dict(
            color="orange",
            dash="dot"
        ),
        name="95% warming simulation interval",
        hovertemplate=(
            "Observation: %{x}<br>"
            "Lower 95% simulation interval: %{y:,.0f} tons"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# ------------------------------------------------------------
# Panel 2: Percentage difference from baseline
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_warm["Biomass_change_pct"],
        mode="lines+markers",
        name="% Change",
        line=dict(color="red"),
        customdata=np.column_stack(
            (
                warming_biomass,
                baseline_slice,
                df_warm["Year"],
                df_warm["Month"]
            )
        ),
        hovertemplate=(
            "Observation: %{x}<br>"
            "Biomass change: %{y:.2f}%<br>"
            "Warming biomass: %{customdata[0]:,.0f} tons<br>"
            "Baseline biomass: %{customdata[1]:,.0f} tons<br>"
            "Year: %{customdata[2]:.0f}<br>"
            "Month: %{customdata[3]:.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    row=2,
    col=1
)

fig.add_hline(
    y=5,
    line_dash="dot",
    line_color="gray",
    row=2,
    col=1
)

fig.add_hline(
    y=-5,
    line_dash="dot",
    line_color="gray",
    row=2,
    col=1
)

# ------------------------------------------------------------
# Panel 3: Thermal suitability and effort
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=month_x,
        y=df_warm["EnvIndex"],
        mode="lines+markers",
        name="Thermal Suitability",
        line=dict(color="green"),
        customdata=np.column_stack(
            (
                df_warm["SST"],
                np.full(len(df_warm), T_opt),
                df_warm["SST"] - T_opt,
                df_warm["Year"],
                df_warm["Month"]
            )
        ),
        hovertemplate=(
            "Observation: %{x}<br>"
            "Thermal suitability: %{y:.3f}<br>"
            "Warmed SST: %{customdata[0]:.2f}°C<br>"
            "Thermal optimum: %{customdata[1]:.2f}°C<br>"
            "Distance from optimum: %{customdata[2]:+.2f}°C<br>"
            "Year: %{customdata[3]:.0f}<br>"
            "Month: %{customdata[4]:.0f}"
            "<extra></extra>"
        )
    ),
    row=3,
    col=1
)

fig.add_trace(
    go.Bar(
        x=month_x,
        y=df_warm["Effort"],
        name="Effort (vessel-days)",
        marker=dict(
            color="rgba(0,0,255,0.30)"
        ),
        customdata=np.column_stack(
            (
                df_warm["TotalCatch_tons"],
                df_warm["CPUE_tons"],
                df_warm["Year"],
                df_warm["Month"]
            )
        ),
        hovertemplate=(
            "Observation: %{x}<br>"
            "Effort: %{y:.0f} vessel-days<br>"
            "Observed catch: %{customdata[0]:,.0f} tons<br>"
            "Observed CPUE: %{customdata[1]:,.2f} tons/vessel-day<br>"
            "Year: %{customdata[2]:.0f}<br>"
            "Month: %{customdata[3]:.0f}"
            "<extra></extra>"
        )
    ),
    row=3,
    col=1
)

fig.update_layout(
    height=920,
    showlegend=True,
    title=(
        f"🔥 Warming Scenario (+{delta_T:.1f}°C) "
        f"— catchability q={q:.2e}"
    ),
    template="plotly_dark",
    hovermode="x unified"
)

fig.update_yaxes(
    title_text="Biomass (tons)",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="% Change",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Thermal suitability / effort",
    row=3,
    col=1
)

fig.update_xaxes(
    title_text="Time sequence",
    row=3,
    col=1
)

fig.show()

output_path = (
    output_dir
    / "biomass_scenarios_comparison.png"
)

fig.write_image(
    str(output_path)
)

print(f"Saved plot to: {output_path}")


aavg_pct_change = float(
    df_warm["Biomass_change_pct"].mean()
)

final_change = float(
    df_warm["Biomass_change_pct"].iloc[-1]
)

baseline_thermal_mean = float(
    df_mc["ThermalSuitability"]
    .iloc[:duration]
    .mean()
)

warmed_thermal_mean = float(
    df_warm["ThermalSuitability"].mean()
)

thermal_suitability_change = (
    100
    * (
        warmed_thermal_mean
        - baseline_thermal_mean
    )
    / (baseline_thermal_mean + 1e-12)
)

print(
    "Average biomass change under warming: "
    f"{avg_pct_change:.2f}%"
)

print(
    "Final biomass change under warming: "
    f"{final_change:.2f}%"
)

print(
    "Average thermal-suitability change: "
    f"{thermal_suitability_change:.2f}%"
)

Saved plot to: ..\outputs\EDSPM\biomass_scenarios_comparison.png
Average biomass change under warming: 0.70%
Final biomass change under warming: 5.16%
Average thermal-suitability change: 2.83%


## 2️⃣ Warming Simulation Insights

The warming figure compares the modelled biomass trajectory under a gradual **+2°C scenario** with the corresponding baseline over the selected 24-observation period.

Both scenarios begin near **3.0 million tonnes**. Under the default configuration, the baseline rises to approximately **3.07 million tonnes**, while the warming trajectory reaches approximately **3.20 million tonnes**.

The warming scenario therefore finishes about **130,000 tonnes**, or roughly **4%**, above the corresponding baseline endpoint. This represents a moderate positive warming effect under the selected assumptions.

### Panel interpretation

1. **Simulated biomass**
   The first panel shows baseline mean biomass, warming mean biomass, and the central 95% warming simulation interval.

   Both trajectories increase, but the warming trajectory rises more strongly and becomes visibly separated from the baseline. The warming line increases by approximately 6–7% from its own starting value, although only the difference relative to baseline should be interpreted as the warming-specific effect.

2. **Percentage change from baseline**
   The second panel isolates the warming effect by reporting the percentage difference between warming biomass and the corresponding baseline biomass.

   Under the default settings, this difference develops progressively and reaches approximately **4% near the end of the scenario period**. This panel is the most appropriate place to assess the effect attributable specifically to warming.

3. **Thermal suitability and effort**
   The third panel shows thermal suitability alongside fishing effort.

   Under the default +2°C scenario, warming moves temperatures closer to the assumed thermal optimum during part of the selected period. This strengthens the modelled temperature-dependent growth response and helps explain why warming biomass rises above baseline.

   Fishing effort remains visible as a separate removal pressure. Under the default catchability setting, fishing removals are not large enough to offset the additional modelled growth.

### Interpretation boundaries

Warming does not have a universally positive or negative effect in this model. The response depends on:

* starting SST;
* the assumed thermal optimum;
* the magnitude and duration of warming;
* carrying capacity;
* catchability;
* fishing effort;
* and the simplified model structure.

Where warming moves temperatures closer to the assumed optimum, modelled growth may increase. Where additional warming moves temperatures beyond the optimum, modelled growth may decline.

The default result therefore represents a **conditional favourable-warming window**, not evidence that climate warming is generally beneficial to *Illex argentinus*.

The numerical interpretation should be taken from the current code output. Fixed statements such as “warming always increases biomass by 1–2%” should not be used because the result changes with parameter values and scenario settings.

### Decision-support value

The comparison supports scenario screening by showing:

* the absolute biomass trajectory under warming;
* the additional effect relative to baseline;
* the thermal mechanism associated with that response;
* and the fishing effort occurring over the same period.

Under the default configuration, the +2°C scenario produces a moderate positive biomass response relative to baseline. This is a conditional model result, not a climate forecast, stock-assessment conclusion, or direct management prescription.


### Sensitivity Analysis: CPUE versus Modelled Biomass

Sensitivity Analysis: CPUE versus Modelled Biomass

This section evaluates how closely observed CPUE follows the biomass trajectories generated by the baseline and warming simulations.

The same workflow is applied separately to:

* the complete baseline biomass series; and
* the selected-duration warming biomass series.

The analysis:

1. applies an illustrative 10% observation-noise assumption to CPUE;
2. calculates mean CPUE and central simulation bounds;
3. normalizes CPUE and modelled biomass to a 0–1 scale;
4. calculates Pearson correlation for each scenario;
5. creates a time-series comparison of CPUE and biomass;
6. creates a scatter plot with a linear association trend;
7. and reports slope, intercept, Pearson correlation, and (R^2) in the trend-line hover information.

Under the default configuration, the scatter plots show a weak or negative relationship between CPUE and modelled biomass. This suggests potential decoupling between catch rates and the broad biomass signal represented by the EDSPM.

The result does not prove that CPUE is unrelated to true abundance. It shows that CPUE does not closely track the modelled biomass trajectory under the selected assumptions and observation periods.

The linear trend summarizes the association between CPUE and the output of the nonlinear EDSPM. It is not the EDSPM growth function and does not imply that the underlying population model is linear.

The CPUE uncertainty bands are illustrative and are not independently estimated survey-confidence intervals.

In [23]:
# ============================================================
# CPUE VS BIOMASS — BASELINE AND WARMING SCENARIOS
# Mirrors the Sensitivity & CPUE tab in the Streamlit app
# ============================================================

def prepare_cpue_biomass_comparison(
    scenario_df,
    scenario_label,
    cpue_noise_pct=0.10,
    n_sim_cpue=500,
    random_seed=42
):
    """
    Prepare CPUE uncertainty, normalized indices, correlation,
    and plotting data for one biomass scenario.

    Parameters
    ----------
    scenario_df : pandas.DataFrame
        Baseline or warming simulation results.
    scenario_label : str
        Human-readable scenario name.
    cpue_noise_pct : float
        Illustrative CPUE observation-noise assumption.
    n_sim_cpue : int
        Number of CPUE Monte Carlo simulations.
    random_seed : int
        Seed used to make notebook results reproducible.
    """

    df_latest = scenario_df.copy().reset_index(drop=True)

    required_columns = [
        "Year",
        "Month",
        "CPUE_tons",
        "Biomass_mean",
        "Biomass_CI_lower",
        "Biomass_CI_upper"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df_latest.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{scenario_label} is missing required columns: "
            f"{missing_columns}"
        )

    # Create a proper date column
    df_latest["Date"] = pd.to_datetime(
        df_latest["Year"].astype(str)
        + "-"
        + df_latest["Month"].astype(str)
        + "-01"
    )

    # --------------------------------------------------------
    # Illustrative CPUE uncertainty
    # --------------------------------------------------------

    rng_cpue = np.random.default_rng(random_seed)

    cpue_sim = np.zeros(
        (len(df_latest), n_sim_cpue)
    )

    cpue_values = df_latest[
        "CPUE_tons"
    ].to_numpy()

    for i in range(n_sim_cpue):
        cpue_sim[:, i] = (
            cpue_values
            * (
                1
                + rng_cpue.normal(
                    0,
                    cpue_noise_pct,
                    size=len(df_latest)
                )
            )
        )

    df_latest["CPUE_mean"] = (
        cpue_sim.mean(axis=1)
    )

    df_latest["CPUE_CI_upper"] = np.percentile(
        cpue_sim,
        97.5,
        axis=1
    )

    df_latest["CPUE_CI_lower"] = np.percentile(
        cpue_sim,
        2.5,
        axis=1
    )

    # --------------------------------------------------------
    # Safe normalization
    # --------------------------------------------------------

    def safe_normalize(series):
        min_value = series.min()
        max_value = series.max()

        if (
            pd.isna(min_value)
            or pd.isna(max_value)
            or max_value == min_value
        ):
            return pd.Series(
                0.5,
                index=series.index,
                dtype=float
            )

        return (
            (series - min_value)
            / (max_value - min_value)
        )

    df_latest["Biomass_mean_index"] = safe_normalize(
        df_latest["Biomass_mean"]
    )

    df_latest["Biomass_CI_upper_index"] = safe_normalize(
        df_latest["Biomass_CI_upper"]
    )

    df_latest["Biomass_CI_lower_index"] = safe_normalize(
        df_latest["Biomass_CI_lower"]
    )

    df_latest["CPUE_mean_index"] = safe_normalize(
        df_latest["CPUE_mean"]
    )

    df_latest["CPUE_CI_upper_index"] = safe_normalize(
        df_latest["CPUE_CI_upper"]
    )

    df_latest["CPUE_CI_lower_index"] = safe_normalize(
        df_latest["CPUE_CI_lower"]
    )

    # --------------------------------------------------------
    # Correlation
    # --------------------------------------------------------

    valid_corr = df_latest[
        [
            "CPUE_mean_index",
            "Biomass_mean_index"
        ]
    ].dropna()

    if len(valid_corr) >= 3:
        correlation = valid_corr[
            "CPUE_mean_index"
        ].corr(
            valid_corr[
                "Biomass_mean_index"
            ]
        )
    else:
        correlation = np.nan

    if pd.isna(correlation):
        strength = "uncertain"
        direction = "uncertain"

    elif abs(correlation) >= 0.7:
        strength = "strong"
        direction = (
            "positive"
            if correlation > 0
            else "negative"
        )

    elif abs(correlation) >= 0.4:
        strength = "moderate"
        direction = (
            "positive"
            if correlation > 0
            else "negative"
        )

    else:
        strength = "weak"
        direction = (
            "positive"
            if correlation > 0
            else "negative"
        )

    return (
        df_latest,
        correlation,
        strength,
        direction
    )


def plot_cpue_biomass_comparison(
    scenario_df,
    scenario_label,
    file_prefix
):
    """
    Create the same two CPUE-versus-biomass plots used by the app:
    1. normalized time-series comparison;
    2. scatter relationship with trend line.
    """

    (
        df_latest,
        correlation,
        strength,
        direction
    ) = prepare_cpue_biomass_comparison(
        scenario_df=scenario_df,
        scenario_label=scenario_label
    )

    correlation_label = (
        f"{correlation:.2f}"
        if not pd.isna(correlation)
        else "NA"
    )

    # ========================================================
    # Plot 1: Time-series comparison
    # ========================================================

    fig = go.Figure()

    # CPUE mean
    fig.add_trace(
        go.Scatter(
            x=df_latest["Date"],
            y=df_latest["CPUE_mean_index"],
            mode="lines+markers",
            name="CPUE (mean)",
            line=dict(
                color="#00BFFF",
                width=2
            ),
            customdata=np.column_stack(
                (
                    df_latest["CPUE_mean"],
                    df_latest["CPUE_tons"],
                    df_latest["Year"],
                    df_latest["Month"]
                )
            ),
            hovertemplate=(
                "Date: %{x|%Y-%m}<br>"
                "CPUE index: %{y:.3f}<br>"
                "CPUE mean: %{customdata[0]:,.2f} tons/vessel-day<br>"
                "Observed CPUE: %{customdata[1]:,.2f} tons/vessel-day<br>"
                "Year: %{customdata[2]:.0f}<br>"
                "Month: %{customdata[3]:.0f}"
                "<extra></extra>"
            )
        )
    )

    # CPUE upper interval
    fig.add_trace(
        go.Scatter(
            x=df_latest["Date"],
            y=df_latest["CPUE_CI_upper_index"],
            mode="lines",
            line=dict(
                color="#00BFFF",
                dash="dot"
            ),
            showlegend=False,
            hovertemplate=(
                "Date: %{x|%Y-%m}<br>"
                "Upper CPUE interval index: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )

    # CPUE lower interval and ribbon
    fig.add_trace(
        go.Scatter(
            x=df_latest["Date"],
            y=df_latest["CPUE_CI_lower_index"],
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(0,191,255,0.2)",
            line=dict(
                color="#00BFFF",
                dash="dot"
            ),
            showlegend=False,
            hovertemplate=(
                "Date: %{x|%Y-%m}<br>"
                "Lower CPUE interval index: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )

    # Biomass mean
    fig.add_trace(
        go.Scatter(
            x=df_latest["Date"],
            y=df_latest["Biomass_mean_index"],
            mode="lines+markers",
            name="Biomass (mean)",
            line=dict(
                color="#9932CC",
                dash="dash",
                width=2
            ),
            customdata=np.column_stack(
                (
                    df_latest["Biomass_mean"],
                    df_latest["SST"],
                    df_latest["ChlA"],
                    df_latest["Year"],
                    df_latest["Month"]
                )
            ),
            hovertemplate=(
                "Date: %{x|%Y-%m}<br>"
                "Biomass index: %{y:.3f}<br>"
                "Biomass mean: %{customdata[0]:,.0f} tons<br>"
                "SST: %{customdata[1]:.2f}°C<br>"
                "ChlA: %{customdata[2]:.3f} mg/m³<br>"
                "Year: %{customdata[3]:.0f}<br>"
                "Month: %{customdata[4]:.0f}"
                "<extra></extra>"
            )
        )
    )

    # Biomass upper interval
    fig.add_trace(
        go.Scatter(
            x=df_latest["Date"],
            y=df_latest["Biomass_CI_upper_index"],
            mode="lines",
            line=dict(
                color="#9932CC",
                dash="dot"
            ),
            showlegend=False,
            hovertemplate=(
                "Date: %{x|%Y-%m}<br>"
                "Upper biomass interval index: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )

    # Biomass lower interval and ribbon
    fig.add_trace(
        go.Scatter(
            x=df_latest["Date"],
            y=df_latest["Biomass_CI_lower_index"],
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(153,50,204,0.2)",
            line=dict(
                color="#9932CC",
                dash="dot"
            ),
            showlegend=False,
            hovertemplate=(
                "Date: %{x|%Y-%m}<br>"
                "Lower biomass interval index: %{y:.3f}"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=(
            f"{scenario_label}: CPUE vs Biomass Index "
            f"(r = {correlation_label}, "
            f"{strength} {direction} relationship)"
        ),
        xaxis_title="Time",
        yaxis_title="Normalized index (0–1)",
        height=520,
        template="plotly_dark",
        hovermode="x unified",
        legend=dict(
            orientation="h",
            y=-0.25,
            x=0.5,
            xanchor="center"
        )
    )

    fig.show()

    # ========================================================
    # Plot 2: Scatter relationship with linear trend line
    # ========================================================

    valid_scatter = (
        df_latest[
            [
                "Date",
                "Year",
                "Month",
                "CPUE_tons",
                "CPUE_mean",
                "Biomass_mean",
                "Biomass_mean_index",
                "CPUE_mean_index"
            ]
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    scatter_fig = go.Figure()

    # --------------------------------------------------------
    # Scatter observations
    # --------------------------------------------------------

    scatter_customdata = np.column_stack(
        (
            valid_scatter["Date"]
            .dt.strftime("%Y-%m")
            .to_numpy(),

            valid_scatter["Biomass_mean"]
            .to_numpy(),

            valid_scatter["CPUE_tons"]
            .to_numpy(),

            valid_scatter["CPUE_mean"]
            .to_numpy(),

            valid_scatter["Year"]
            .to_numpy(),

            valid_scatter["Month"]
            .to_numpy()
        )
    ).astype(object)

    scatter_fig.add_trace(
        go.Scatter(
            x=valid_scatter["Biomass_mean_index"],
            y=valid_scatter["CPUE_mean_index"],
            mode="markers",
            name="CPUE vs Biomass",
            marker=dict(
                size=7,
                color="#39FF14",
                opacity=0.65,
                line=dict(
                    color="white",
                    width=0.5
                )
            ),
            customdata=scatter_customdata,
            hovertemplate=(
                "Scenario observation<br>"
                "Date: %{customdata[0]}<br>"
                "Biomass index: %{x:.3f}<br>"
                "CPUE index: %{y:.3f}<br>"
                "Biomass mean: %{customdata[1]:,.0f} tons<br>"
                "Observed CPUE: %{customdata[2]:,.2f} tons/vessel-day<br>"
                "Simulated CPUE mean: %{customdata[3]:,.2f} tons/vessel-day<br>"
                "Year: %{customdata[4]:.0f}<br>"
                "Month: %{customdata[5]:.0f}"
                "<extra></extra>"
            )
        )
    )

    # --------------------------------------------------------
    # Linear trend line
    # --------------------------------------------------------

    n_valid = len(valid_scatter)

    n_unique_biomass = valid_scatter[
        "Biomass_mean_index"
    ].nunique()

    if n_valid >= 3 and n_unique_biomass > 1:

        x_values = valid_scatter[
            "Biomass_mean_index"
        ].to_numpy(dtype=float)

        y_values = valid_scatter[
            "CPUE_mean_index"
        ].to_numpy(dtype=float)

        slope, intercept = np.polyfit(
            x_values,
            y_values,
            1
        )

        x_line = np.linspace(
            x_values.min(),
            x_values.max(),
            200
        )

        y_line = (
            slope * x_line
            + intercept
        )

        # Predicted values and basic fit statistic
        y_predicted = (
            slope * x_values
            + intercept
        )

        residual_sum_squares = np.sum(
            (y_values - y_predicted) ** 2
        )

        total_sum_squares = np.sum(
            (y_values - y_values.mean()) ** 2
        )

        r_squared = (
            1 - residual_sum_squares / total_sum_squares
            if total_sum_squares > 0
            else np.nan
        )

        trend_customdata = np.column_stack(
            (
                np.full(
                    len(x_line),
                    slope
                ),
                np.full(
                    len(x_line),
                    intercept
                ),
                np.full(
                    len(x_line),
                    correlation
                ),
                np.full(
                    len(x_line),
                    r_squared
                )
            )
        )

        scatter_fig.add_trace(
            go.Scatter(
                x=x_line,
                y=y_line,
                mode="lines",
                name="Linear trend line",
                line=dict(
                    color="#FFD700",
                    dash="dot",
                    width=4
                ),
                customdata=trend_customdata,
                hovertemplate=(
                    "Linear trend line<br>"
                    "Biomass index: %{x:.3f}<br>"
                    "Predicted CPUE index: %{y:.3f}<br>"
                    "Slope: %{customdata[0]:.3f}<br>"
                    "Intercept: %{customdata[1]:.3f}<br>"
                    "Pearson r: %{customdata[2]:.3f}<br>"
                    "R²: %{customdata[3]:.3f}"
                    "<extra></extra>"
                )
            )
        )

    else:
        print(
            f"Trend line not created for {scenario_label}: "
            f"{n_valid} valid observations and "
            f"{n_unique_biomass} unique biomass values."
        )

    # --------------------------------------------------------
    # Scatter layout
    # --------------------------------------------------------

    scatter_fig.update_layout(
        title=(
            f"{scenario_label}: Scatter Relationship "
            f"between CPUE and Biomass "
            f"(r = {correlation_label})"
        ),
        xaxis_title="Biomass index (0–1)",
        yaxis_title="CPUE index (0–1)",
        template="plotly_dark",
        height=450,
        hovermode="closest",
        legend=dict(
            orientation="h",
            y=-0.22,
            x=0.5,
            xanchor="center"
        )
    )

    scatter_fig.update_xaxes(
        range=[-0.03, 1.03]
    )

    scatter_fig.update_yaxes(
        range=[-0.03, 1.03]
    )

    scatter_fig.show()
    # ========================================================
    # Save both plots
    # ========================================================

    time_series_path = (
        output_dir
        / f"{file_prefix}_cpue_vs_biomass_comparison.png"
    )

    scatter_path = (
        output_dir
        / f"{file_prefix}_cpue_vs_biomass_scatter.png"
    )

    try:
        fig.write_image(
            str(time_series_path)
        )

        scatter_fig.write_image(
            str(scatter_path)
        )

        print(
            f"Saved {scenario_label} plots:\n"
            f"- {time_series_path}\n"
            f"- {scatter_path}"
        )

    except Exception as error:
        print(
            f"Static image export skipped for "
            f"{scenario_label}: {error}"
        )

    # ========================================================
    # Scenario interpretation
    # ========================================================

    print(
        f"\n{scenario_label} interpretation:"
    )

    if pd.isna(correlation):
        print(
            "The CPUE–biomass relationship could not "
            "be estimated reliably."
        )

    elif correlation > 0.5:
        print(
            "CPUE and modelled biomass generally move "
            "together under this scenario."
        )

    elif correlation < -0.5:
        print(
            "CPUE and modelled biomass diverge under "
            "this scenario."
        )

    else:
        print(
            "The relationship is weak. Fleet behaviour, "
            "aggregation, migration timing, environmental "
            "hotspots, catchability, or model limitations "
            "may contribute to the mismatch."
        )

    return {
        "scenario": scenario_label,
        "correlation": correlation,
        "strength": strength,
        "direction": direction,
        "data": df_latest,
        "time_series_figure": fig,
        "scatter_figure": scatter_fig
    }


# ============================================================
# RUN BOTH SCENARIOS
# ============================================================

baseline_cpue_results = plot_cpue_biomass_comparison(
    scenario_df=df_mc,
    scenario_label="Baseline Simulation",
    file_prefix="baseline"
)

warming_cpue_results = plot_cpue_biomass_comparison(
    scenario_df=df_warm,
    scenario_label=f"Warming Scenario (+{delta_T:.1f}°C)",
    file_prefix="warming"
)

Saved Baseline Simulation plots:
- ..\outputs\EDSPM\baseline_cpue_vs_biomass_comparison.png
- ..\outputs\EDSPM\baseline_cpue_vs_biomass_scatter.png

Baseline Simulation interpretation:
The relationship is weak. Fleet behaviour, aggregation, migration timing, environmental hotspots, catchability, or model limitations may contribute to the mismatch.


Saved Warming Scenario (+2.0°C) plots:
- ..\outputs\EDSPM\warming_cpue_vs_biomass_comparison.png
- ..\outputs\EDSPM\warming_cpue_vs_biomass_scatter.png

Warming Scenario (+2.0°C) interpretation:
The relationship is weak. Fleet behaviour, aggregation, migration timing, environmental hotspots, catchability, or model limitations may contribute to the mismatch.


---

## 📘 CPUE versus Modelled Biomass: Baseline and Warming Scenarios

> **Seasonal limitation:** The available dataset consistently covers January–June. Baseline results use the complete available sequence, while the default warming comparison uses the selected warming duration. These results may not represent full-year stock behaviour.

### 1️⃣ What the Figures Show

For each scenario, the notebook produces:

- a normalized time-series comparison of CPUE and modelled biomass; and
- a scatter plot showing their direct statistical association.

The analysis asks:

> How closely does observed CPUE track biomass generated by the EDSPM under the selected scenario assumptions?

It does not test CPUE against independently measured true biomass.

---

### 2️⃣ Interpreting the Time-Series Figure

The time-series plot helps identify whether CPUE and modelled biomass rise and fall together.

A mismatch may occur because CPUE can be influenced by:

- fleet targeting;
- changes in catchability;
- aggregation and schooling;
- migration timing;
- environmental fronts and hotspots;
- spatial mismatch between fishing activity and the represented biomass process;
- and limitations in the biomass model itself.

The hover information provides normalized indices together with actual CPUE, modelled biomass, SST, chlorophyll-a, date, year, and month.

---

### 3️⃣ Interpreting the Scatter Plot

The scatter plot compares normalized biomass with normalized CPUE.

The gold dotted line is a **linear association summary**, not the nonlinear EDSPM curve.

Its hover information reports:

- predicted CPUE index;
- slope;
- intercept;
- Pearson correlation, `r`;
- and coefficient of determination, `R²`.

Pearson `r` describes the direction and strength of the linear association. `R²` describes the proportion of CPUE-index variation represented by that fitted linear relationship.

A shallow or nearly flat trend suggests weak linear association. A positive or negative slope indicates the direction of the relationship, but neither establishes causation.

---

### 4️⃣ Scenario-Specific Interpretation

The correlation and fitted trend should be interpreted separately for:

- **Baseline Simulation**
- **Warming Scenario**

The warming analysis may produce a different correlation because it uses a different biomass trajectory and typically a shorter observation window.

The notebook therefore avoids hard-coded claims such as a fixed correlation of −0.11. The current correlation, direction, strength, and R² should be read directly from each figure.

---

### 5️⃣ Decision-Support Interpretation

The weak or negative relationship highlights potential decoupling between CPUE and simulated biomass.

The principal management implication is:

CPUE should not be used as the sole proxy for stock abundance.

A defensible monitoring framework should interpret CPUE alongside:

* independent survey or abundance information;
* environmental conditions;
* fishing effort;
* fleet behaviour;
* biological knowledge;
* spatial distribution;
* and model-based scenario outputs.

---

## ✅ Combined Summary

### Baseline Simulation

- Biomass increases from approximately 3.0 million to around 3.6 million tonnes across the complete available observation sequence.
- The trajectory reflects nonlinear temperature-dependent growth, chlorophyll-a productivity, carrying-capacity constraints, catchability, effort, and stochastic growth variation.
- The simulation interval represents assumption-driven Monte Carlo variability, not a formal confidence interval.

### Warming Scenario

- The default (+2^\circ\text{C}) scenario produces a stronger biomass trajectory than the corresponding baseline.
- Warming biomass rises from approximately 3.0 million to around 3.20 million tonnes, while the equivalent baseline reaches approximately 3.07 million tonnes.
- The warming trajectory therefore finishes roughly 4% above the corresponding baseline.
- The positive response occurs because warming moves temperatures closer to the assumed thermal optimum during part of the selected period.
- This represents a conditional favourable-warming window, not evidence that climate warming is universally beneficial.

### CPUE versus Modelled Biomass

- CPUE shows a weak or negative association with modelled biomass under the default configuration.
- Catch rates and model outputs therefore provide different signals across the analysed observations.
- The analysis does not compare CPUE with independently measured true abundance.
- The results support using multiple indicators rather than CPUE-only stock interpretation.

### Final Decision-Support Takeaway

No single indicator should be used alone to infer stock condition.

For a short-lived and migratory species such as *Illex argentinus*, CPUE should be considered together with environmental conditions, effort, fleet behaviour, survey information, biological knowledge, and transparent scenario modelling.

The EDSPM provides an additional environment-informed scenario signal. It does not replace formal stock assessment, validated biomass surveys, or expert fisheries review.

---

### Starting-Biomass Plausibility Check

A simple screening rule checks whether the assumed starting biomass is large enough to support the largest observed catch under an illustrative target exploitation fraction:

$
N_0 \geq \frac{\text{maximum observed catch}}{E_{\text{target}}}
$

The notebook calculates the maximum observed catch directly from the loaded dataset and applies an illustrative target fraction of 30%.

This is a lower-bound plausibility check only. It does not estimate true biomass, establish a sustainable exploitation rate, or constitute formal model calibration.

In [17]:
# compute example using actual data
max_catch = df["TotalCatch_tons"].max()
E_target = 0.30
N0_min = max_catch / E_target
print(f"Max catch = {max_catch:,.0f} tons. For E_target={E_target:.2%}, N0 >= {N0_min:,.0f} tons.")

Max catch = 100,221 tons. For E_target=30.00%, N0 >= 334,070 tons.


### Catchability (`q`) in the Model

Catchability represents the efficiency with which fishing effort removes available biomass.

The model applies catchability through:

$
H_t = q \times Effort_t \times N_t
$

Where:

- `H_t` is modelled harvest;
- `q` is the catchability coefficient;
- `Effort_t` is vessel-day effort;
- and `N_t` is modelled biomass.

A higher `q` increases fishing removals for the same effort and biomass. A lower `q` produces lighter modelled harvest pressure.

Under the default value of $(q = 2 \times 10^{-4})$, modelled fishing pressure remains relatively light compared with simulated biomass growth.

Catchability is already available as an adjustable parameter in the app and is retained as a notebook parameter for scenario testing. It should be interpreted as a simplified scenario coefficient rather than a formally estimated fleet-catchability parameter.